# One grammar: LangGraph + Agentic Systems · 2.1.4

Declare state, nodes and edges with LangGraph. Agent organizes operations; System coordinates units; Environment executes a graph for each observation. All use graph=Graph(...). compile validates and prepares; run executes and returns RunResult.

This notebook runs offline without model calls or server startup. Install local wheel 2.1.4 first. Historical dev1 live evidence does not certify this artifact.

Parallel branches write their own fields. An edge from a list of nodes is a barrier: the next node waits for all. There is no separate composition constructor or toolkit merge callback.

In [2]:
import pydantic
from pathlib import Path
import sys
import agentic_systems as toolkit

assert toolkit.__version__ == '2.1.4'
ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "examples/common_graph/application.py").is_file()
    )
)
sys.path.insert(0, str(ROOT))


class State(pydantic.BaseModel):
    value: int
    increased: int | None = None
    doubled: int | None = None


def increase(state):
    return {"value": state.value + 1}


def double(state):
    return {"value": state.value * 2}

## Agent: two operations in order

In [3]:
graph = toolkit.Graph(State)
graph.add_node("increase", increase)
graph.add_node("double", double)
graph.add_edge(toolkit.START, "increase")
graph.add_edge("increase", "double")
graph.add_edge("double", toolkit.END)
agent = toolkit.Agent("arithmetic", graph=graph)
result = agent.run({"value": 2})
assert result.ok and result.data["value"] == 6
assert result.check_invariants().ok
print(result.human_results())

arithmetic: COMPLETED
Execution: run-c8af5490659d4a31bdbb5e6e0df70786
Input: {"value":2}
Output: {"value":6}
Process:
- Step increase: completed
- Step double: completed


## System: coordinate two Agents

Each Agent has its own graph and is used directly as a System node.

In [4]:
increase_graph = toolkit.Graph(State)
increase_graph.add_node("increase", increase)
increase_graph.add_edge(toolkit.START, "increase")
increase_graph.add_edge("increase", toolkit.END)
incrementer = toolkit.Agent("incrementer", graph=increase_graph)
double_graph = toolkit.Graph(State)
double_graph.add_node("double", double)
double_graph.add_edge(toolkit.START, "double")
double_graph.add_edge("double", toolkit.END)
multiplier = toolkit.Agent("multiplier", graph=double_graph)
graph = toolkit.Graph(State)
graph.add_node("incrementer", incrementer)
graph.add_node("multiplier", multiplier)
graph.add_edge(toolkit.START, "incrementer")
graph.add_edge("incrementer", "multiplier")
graph.add_edge("multiplier", toolkit.END)
system = toolkit.System("team", graph=graph)
result = system.run({"value": 2})
assert result.ok and result.data["value"] == 6
assert result.check_invariants().ok
print(result.human_results())

team: COMPLETED
Execution: run-3bf546f3462942688bf4db54765c04cd
Input: {"value":2}
Output: {"value":6}
Process:
- Agent incrementer: completed
- Step increase: completed
- Agent multiplier: completed
- Step double: completed


## Agent: concurrent branches

Both read value=2. One writes increased=3; the other doubled=4. The barrier enables total after both finish. Each node returns only its updates.

In [5]:
def left(state):
    return {"increased": state.value + 1}


def right(state):
    return {"doubled": state.value * 2}


def total(state):
    return {"value": state.increased + state.doubled}


graph = toolkit.Graph(State)
graph.add_node("left", left)
graph.add_node("right", right)
graph.add_node("total", total)
graph.add_edge(toolkit.START, "left")
graph.add_edge(toolkit.START, "right")
graph.add_edge(["left", "right"], "total")
graph.add_edge("total", toolkit.END)
agent = toolkit.Agent("concurrent", graph=graph)
result = agent.run({"value": 2})
assert result.ok and result.data["value"] == 7
assert result.check_invariants().ok
print(result.human_results())

concurrent: COMPLETED
Execution: run-bd2125e3f0c8489ca4384caa52443f2e
Input: {"value":2}
Output: {"value":7,"increased":3,"doubled":4}
Process:
- Step left: completed
- Step right: completed
- Step total: completed


## System: concurrent Agents

The same LangGraph mechanism applies. Agents used as nodes propagate fields they wrote without returning every received field.

In [6]:
left_graph = toolkit.Graph(State)
left_graph.add_node("left", left)
left_graph.add_edge(toolkit.START, "left")
left_graph.add_edge("left", toolkit.END)
left_agent = toolkit.Agent("left", graph=left_graph)
right_graph = toolkit.Graph(State)
right_graph.add_node("right", right)
right_graph.add_edge(toolkit.START, "right")
right_graph.add_edge("right", toolkit.END)
right_agent = toolkit.Agent("right", graph=right_graph)
graph = toolkit.Graph(State)
graph.add_node("left", left_agent)
graph.add_node("right", right_agent)
graph.add_node("total", total)
graph.add_edge(toolkit.START, "left")
graph.add_edge(toolkit.START, "right")
graph.add_edge(["left", "right"], "total")
graph.add_edge("total", toolkit.END)
system = toolkit.System("concurrent-team", graph=graph)
result = system.run({"value": 2})
assert result.ok and result.data["value"] == 7
assert result.check_invariants().ok
print(result.human_results())

concurrent-team: COMPLETED
Execution: run-14128a30bc284071b6976dbe78320774
Input: {"value":2}
Output: {"value":7,"increased":3,"doubled":4}
Process:
- Agent left: completed
- Step left: completed
- Agent right: completed
- Step right: completed
- Step total: completed


## Agent: decision

A routing function returns a node name or END. No additional routing object is needed.

In [7]:
graph = toolkit.Graph(State)
graph.add_node("increase", increase)
graph.add_node("double", double)
graph.add_edge(toolkit.START, "increase")
graph.add_conditional_edges(
    "increase",
    lambda state: "double" if state.value >= 0 else toolkit.END,
    ["double", toolkit.END],
)
graph.add_edge("double", toolkit.END)
agent = toolkit.Agent("decision", graph=graph)
result = agent.run({"value": 2})
assert result.ok and result.data["value"] == 6
assert result.check_invariants().ok
print(result.human_results())

decision: COMPLETED
Execution: run-9600e71909954f59a1540d60c04ef4bb
Input: {"value":2}
Output: {"value":6}
Process:
- Step increase: completed
- Decision increase -> "double"
- Step double: completed


## System: decision

A routing function returns a node name or END. No additional routing object is needed.

In [8]:
graph = toolkit.Graph(State)
graph.add_node("increase", incrementer)
graph.add_node("double", multiplier)
graph.add_edge(toolkit.START, "increase")
graph.add_conditional_edges(
    "increase",
    lambda state: "double" if state.value >= 0 else toolkit.END,
    ["double", toolkit.END],
)
graph.add_edge("double", toolkit.END)
system = toolkit.System("decision", graph=graph)
result = system.run({"value": 2})
assert result.ok and result.data["value"] == 6
assert result.check_invariants().ok
print(result.human_results())

decision: COMPLETED
Execution: run-eeb4a30b99cb44dda6d827ed671774c6
Input: {"value":2}
Output: {"value":6}
Process:
- Agent incrementer: completed
- Step increase: completed
- Decision increase -> "double"
- Agent multiplier: completed
- Step double: completed


## Context: domain information separate from state

Context(data, sources=...) is an immutable JSON snapshot. Agent, System and Environment accept context= and pass it to children. Conflicting keys are rejected rather than silently overwritten.

- context.data: rules, vocabulary and working-domain data.
- context.sources: provenance identifiers.
- context.episodes: Memory episodes selected at unit entry.
- context.lineage: at most eight compact observations of preceding evidence; observations are not authority.

Tools receive it through `*, context: toolkit.Context`, not as an argument invented by a model. Skills group capabilities/instructions; Context supplies reference information. State retains working data. Policy is not automatically copied into state.

In [9]:
domain = toolkit.Context(
    {"currency": "MXN", "tax_percent": 16}, sources=("tax-policy-v1",)
)
changed_copy = domain.data
changed_copy["tax_percent"] = 99
assert domain.data["tax_percent"] == 16
print("Context:", domain.to_dict())
print("Fingerprint:", domain.fingerprint)

Context: {'data': {'currency': 'MXN', 'tax_percent': 16}, 'sources': ['tax-policy-v1'], 'episodes': [], 'lineage': []}
Fingerprint: 9458b62c01ac3d63c4695cd67069372d6c4372e74b029d40201194abb01402af


## load_skill: load a capability and use it in a graph

The folder contains SKILL.md, skill.py and tools.py. build_skill() declares the runtime capability; SKILL.md documents it. Loading executes local Python, so its origin must be trusted.

Input 100 MXN → load Skill → quote Tool consults Context → output 116 MXN. Memory supplies the previous episode on the second execution.

In [10]:
class QuoteState(pydantic.BaseModel):
    subtotal: int
    total: int | None = None
    currency: str | None = None
    policy_source: str | None = None
    previous_episodes: int | None = None
    preceding_events: int | None = None


capability = toolkit.load_skill(ROOT / "examples/common_graph/skills/pricing")
graph = toolkit.Graph(QuoteState)
graph.add_node("quote", capability.tools[0])
graph.add_edge(toolkit.START, "quote")
graph.add_edge("quote", toolkit.END)
quoter = toolkit.Agent(
    "quote", graph=graph, skills=(capability,), context=domain, memory=toolkit.Memory()
)
first_quote = quoter.run({"subtotal": 100})
second_quote = quoter.run({"subtotal": 100})
assert first_quote.ok and first_quote.data["total"] == 116
assert second_quote.data["previous_episodes"] == 1
print("Input:", {"subtotal": 100})
print("Process: load_skill → quote(Context) → Memory")
print("Output:", first_quote.data)
print(
    "Previous episodes on the second execution:", second_quote.data["previous_episodes"]
)

Input: {'subtotal': 100}
Process: load_skill → quote(Context) → Memory
Output: {'subtotal': 100, 'total': 116, 'currency': 'MXN', 'policy_source': 'tax-policy-v1', 'previous_episodes': 0, 'preceding_events': 0}
Previous episodes on the second execution: 1


### The same Context in Tools and nodes

Both receive `*, context: toolkit.Context`: the effective unit snapshot with domain data, sources, selected Memory and preceding evidence. Context is not copied into state; Runtime is not imported into it.

In [11]:
def quote_from_reference(state, *, context: toolkit.Context):
    domain = context
    return {
        "total": state.subtotal * (100 + domain.data["tax_percent"]) // 100,
        "currency": domain.data["currency"],
    }


graph = toolkit.Graph(QuoteState)
graph.add_node("quote", quote_from_reference)
graph.add_edge(toolkit.START, "quote")
graph.add_edge("quote", toolkit.END)
contextual_result = toolkit.Agent("runtime-context", graph=graph, context=domain).run(
    {"subtotal": 100}
)
assert contextual_result.ok and contextual_result.data["total"] == 116
print("Input:", {"subtotal": 100}, "→ Runtime.context →", contextual_result.data)

Input: {'subtotal': 100} → Runtime.context → {'subtotal': 100, 'total': 116, 'currency': 'MXN'}


## Project input

Tools can be nodes directly. For differently shaped input, a node function selects fields and calls Tool.run. Projection uses model_dump(exclude_unset=True) to preserve absent values so contracts reject missing required fields.

In [12]:
class NestedState(pydantic.BaseModel):
    order: QuoteState
    total: int | None = None
    currency: str | None = None
    policy_source: str | None = None
    previous_episodes: int | None = None
    preceding_events: int | None = None


graph = toolkit.Graph(NestedState)
def quote_step(state):
    return capability.tools[0].run(state.order.model_dump(mode="json", exclude_unset=True))
graph.add_node(
    "quote",
    quote_step,
)
graph.add_edge(toolkit.START, "quote")
graph.add_edge("quote", toolkit.END)
projected = toolkit.Agent("nested-input", graph=graph, context=domain).run(
    {"order": {"subtotal": 100}}
)
assert projected.ok and projected.data["total"] == 116
print("Input:", {"order": {"subtotal": 100}}, "→ projection →", projected.data)

Input: {'order': {'subtotal': 100}} → projection → {'order': {'subtotal': 100}, 'total': 116, 'currency': 'MXN', 'policy_source': 'tax-policy-v1', 'previous_episodes': 0, 'preceding_events': 0}


## Tools, Skills, Agents, System, Environment and Eval together

R-17 costs 240 MXN and has two components; one is missing. A classifier reads structured evidence; separate Tools compute pricing and policy; a decision requests review; execution pauses and resumes with approval. Environment processes two orders; Eval verifies the contract.

Everything uses Graph. All three SDKs run Python here rather than simulated models. Actor loops remain each SDK’s responsibility; LangGraph coordinates nodes.

In [13]:
"""A refund workflow: Python operators, optional real language interpretation."""
import pydantic
import agentic_systems as toolkit

CASE = {
    "order_id": " R-17 ",
    "amount": 240,
    "expected_items": ["adapter", "sensor"],
    "received_items": ["adapter"],
    "customer_note": "The box arrived sealed. I found the adapter; the sensor on the packing slip was missing. I want to return everything.",
}
SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "order_id",
        "amount",
        "total_units",
        "missing_units",
        "reason",
        "sources",
    ],
    "properties": {
        "order_id": {"type": "string"},
        "amount": {"type": "integer"},
        "total_units": {"type": "integer", "minimum": 1},
        "missing_units": {"type": "integer", "minimum": 0},
        "reason": {"type": "string", "minLength": 5},
        "sources": {"type": "array", "items": {"type": "string"}, "minItems": 1},
    },
}


@toolkit.tool
def normalize(
    order_id: str,
    amount: int,
    expected_items: list,
    received_items: list,
    customer_note: str,
) -> dict:
    """Normalize identifiers and preserve the evidence, without interpretation."""
    if amount < 0 or not expected_items:
        raise ValueError("A case needs nonnegative amount and an itemized packing slip")
    return {
        "order_id": order_id.strip(),
        "amount": amount,
        "expected_items": expected_items,
        "received_items": received_items,
        "customer_note": customer_note,
    }


@toolkit.tool
def classify(
    order_id: str,
    amount: int,
    expected_items: list,
    received_items: list,
    customer_note: str,
) -> dict:
    """Deterministic operator for structured item evidence; no language understanding."""
    return {
        "order_id": order_id,
        "amount": amount,
        "total_units": len(expected_items),
        "missing_units": len(set(expected_items) - set(received_items)),
        "reason": "Difference between packing slip and received item list",
        "sources": ["packing-slip", "customer-report"],
    }


@toolkit.tool
def price(
    order_id: str,
    amount: int,
    total_units: int,
    missing_units: int,
    reason: str,
    sources: list,
) -> dict:
    """Compute money in integer units; never delegate arithmetic to a model."""
    if not 0 <= missing_units <= total_units or total_units < 1:
        raise ValueError("Inconsistent quantities")
    return {"order_id": order_id, "refund": amount * missing_units // total_units}


@toolkit.tool
def policy(
    order_id: str,
    amount: int,
    total_units: int,
    missing_units: int,
    reason: str,
    sources: list,
    *,
    context: toolkit.Context,
) -> dict:
    """Use semantic domain Context and retain the classifier's evidence."""
    return {
        "currency": context.data["currency"],
        "review_threshold": context.data["review_threshold"],
        "reason": reason,
        "sources": sources,
        "missing_units": missing_units,
        "previous_episodes": len(context.episodes),
    }


class RefundState(pydantic.BaseModel):
    order_id: str | None = None
    amount: int | None = None
    expected_items: list | None = None
    received_items: list | None = None
    customer_note: str | None = None
    total_units: int | None = None
    missing_units: int | None = None
    reason: str | None = None
    sources: list | None = None
    refund: int | None = None
    currency: str | None = None
    review_threshold: int | None = None
    previous_episodes: int | None = None
    needs_review: bool | None = None
    status: str | None = None


class RefundOutput(pydantic.BaseModel):
    order_id: str
    refund: int
    currency: str
    missing_units: int
    reason: str
    sources: list
    previous_episodes: int
    status: str


def tool_agent(name, operation, *, skills=()):
    graph = toolkit.Graph(RefundState)
    graph.add_node(name, operation)
    graph.add_edge(toolkit.START, name)
    graph.add_edge(name, toolkit.END)
    return toolkit.Agent(name, graph=graph, skills=skills)


def build_system(framework="native", *, reasoner=None, interactive=False):
    domain = toolkit.Context(
        {
            "currency": "MXN",
            "review_threshold": 100,
            "refund_policy": "Refund only missing items proportionally; the customer's demand is not proof of a fully missing order.",
        },
        sources=("refund-policy-v1", "packing-slip", "customer-report"),
    )
    capability = toolkit.skill(name="refunds", tools=(normalize, price, policy))
    normalizer = tool_agent("normalize", normalize, skills=(capability,))
    classifier = toolkit.Agent(
        "classify",
        runtime=reasoner or toolkit.runtime(provider="python-runtime"),
        framework=toolkit.framework(framework),
        skills=()
        if reasoner
        else (toolkit.skill(name="classification", tools=(classify,)),),
        instructions="Interpret the customer note against packing slip and received items. Apply reference policy. Return only JSON: order_id, amount, total_units, missing_units, short evidence-based reason, sources including packing-slip and customer-report. Copy original amount unchanged; Python calculates refunds."
        if reasoner
        else "",
        contract={"output_schema": SCHEMA},
    )

    def review(data):
        if not interactive:
            return {"status": "pending_review"}
        approved = toolkit.pause(
            {
                "question": "Approve partial refund?",
                "refund": data.refund,
                "order_id": data.order_id,
            }
        )
        if type(approved) is not bool:
            raise TypeError("Approval must be bool")
        return {"status": "approved" if approved else "rejected"}

    graph = toolkit.Graph(RefundState, output_schema=RefundOutput)
    graph.add_node("normalize", normalizer)
    graph.add_node("classify", classifier)
    graph.add_node("price", tool_agent("price", price))
    graph.add_node("policy", tool_agent("policy", policy))
    def decision_step(state):
        return {"needs_review": state.refund > state.review_threshold}
    graph.add_node(
        "decision",
        decision_step,
    )
    graph.add_node("review", review)
    def approve_step(state):
        return {"status": "approved"}
    graph.add_node("approve", approve_step)
    graph.add_edge(toolkit.START, "normalize")
    graph.add_edge("normalize", "classify")
    graph.add_edge("classify", "price")
    graph.add_edge("classify", "policy")
    graph.add_edge(["price", "policy"], "decision")
    graph.add_conditional_edges(
        "decision",
        lambda state: "review" if state.needs_review else "approve",
        ["review", "approve"],
    )
    graph.add_edge("review", toolkit.END)
    graph.add_edge("approve", toolkit.END)
    return toolkit.System(
        "refunds", graph=graph, context=domain, memory=toolkit.Memory()
    )


def check_result(output):
    return (
        output.ok
        and output.data["order_id"] == "R-17"
        and (output.data["refund"] == 120)
        and (output.data["currency"] == "MXN")
        and (output.data["missing_units"] == 1)
        and ({"packing-slip", "customer-report"} <= set(output.data["sources"]))
        and output.check_invariants().ok
    )


def demonstrate(framework="native"):
    definition = build_system(framework, interactive=True)
    program = definition.compile()
    waiting = program.run(CASE, timeout_s=40)
    assert waiting.status == "paused", waiting.to_dict()
    final = program.resume(waiting, True)
    assert check_result(final), final.to_dict()
    graph = toolkit.Graph(RefundState, output_schema=RefundOutput)
    graph.add_node("refund", build_system(framework))
    graph.add_edge(toolkit.START, "refund")
    graph.add_edge("refund", toolkit.END)
    episode = toolkit.Environment(
        "daily-refunds",
        graph=graph,
        records=(CASE, CASE),
        state={"total": 0},
        update=lambda state, row, result: {
            "total": state["total"] + result.data["refund"]
        },
    ).run(timeout_s=40)
    evaluation = toolkit.Eval(
        "refund-contract",
        build_system(framework),
        ({"input": CASE, "expected": {"ok": True}},),
        checks=(lambda result, case: {"ok": check_result(result)},),
    ).run(timeout_s=40)
    assert episode.ok and episode.data["summary"]["state"] == {"total": 240}, (
        episode.to_dict()
    )
    assert evaluation.ok, evaluation.to_dict()
    restored = toolkit.RunResult(**final.to_dict())
    assert restored.check_invariants().ok
    return {
        "orchestration": "langgraph",
        "framework": framework,
        "provider": "python-runtime",
        "input": CASE,
        "process": [
            "normalize",
            "classify",
            "price + policy",
            "decision",
            "pause/resume",
        ],
        "output": final.data,
        "run_results": [
            waiting.to_dict(),
            final.to_dict(),
            episode.to_dict(),
            evaluation.to_dict(),
        ],
        "human_lineage": final.lineage(name="Refund R-17").render(),
        "agent_lineage": final.lineage().to_prompt_context(),
        "memory": definition.memory.read(),
    }


reports = [demonstrate(name) for name in ("native", "openai-agents", "strands")]
for report in reports:
    print(report["framework"], report["output"])

native {'order_id': 'R-17', 'refund': 120, 'currency': 'MXN', 'missing_units': 1, 'reason': 'Difference between packing slip and received item list', 'sources': ['packing-slip', 'customer-report'], 'previous_episodes': 0, 'status': 'approved'}
openai-agents {'order_id': 'R-17', 'refund': 120, 'currency': 'MXN', 'missing_units': 1, 'reason': 'Difference between packing slip and received item list', 'sources': ['packing-slip', 'customer-report'], 'previous_episodes': 0, 'status': 'approved'}
strands {'order_id': 'R-17', 'refund': 120, 'currency': 'MXN', 'missing_units': 1, 'reason': 'Difference between packing slip and received item list', 'sources': ['packing-slip', 'customer-report'], 'previous_episodes': 0, 'status': 'approved'}


## Environment: the same graph parameter

The observation graph contains a System. Environment adds its temporal observe → run → update → observe/END graph. State belongs to the episode; Memory retains observations across runs; Context supplies domain reference.

In [14]:
graph = toolkit.Graph(QuoteState)
graph.add_node("quoter", quoter)
graph.add_edge(toolkit.START, "quoter")
graph.add_edge("quoter", toolkit.END)
episode = toolkit.Environment(
    "quotes",
    graph=graph,
    records=({"subtotal": 100}, {"subtotal": 200}),
    context=domain,
    state={"sum": 0},
    update=lambda state, row, result: {"sum": state["sum"] + result.data["total"]},
)
episode_result = episode.run()
assert episode_result.ok and episode_result.data["summary"]["state"] == {"sum": 348}
print(
    "Input: two quotes → graph per observation →", episode_result.data["summary"]
)
evaluation = toolkit.Eval(
    "tax-contract",
    quoter,
    cases=({"input": {"subtotal": 100}, "expected": {"ok": True}},),
    checks=(lambda result, case: {"ok": result.data["total"] == 116},),
).run()
assert evaluation.ok
print("Eval:", evaluation.data)

Input: two quotes → graph per observation → {'state': {'sum': 348}, 'processed': 2, 'terminated': True, 'truncated': False, 'reason': 'records_exhausted'}


Eval: {'total': 1, 'passed': 1, 'failed': 0, 'cases': [{'name': '0', 'ok': True, 'input': {'subtotal': 100}, 'execution_id': 'run-ed7383195f8342b9976d097bf9f0a128', 'checks': [{'ok': True, 'path': 'ok'}, {'ok': True}], 'judge': None}]}


## Pause, lineage and compact state

State holds working data and at most 32 recent references in the technical _lineage channel. Complete events live outside checkpoints in a flat per-run evidence store. Final output materializes as a RunResult lineage tree; Eval rows and Environment history refer to IDs instead of embedding complete RunResult objects in state.

This avoids duplicating trees at every transition. It does not guarantee constant-size user business data or unlimited storage: external evidence grows with work, bounded by an optional root deadline.

Continuations are process-local. An interrupted function restarts on resume; place earlier effects in separate nodes or make them idempotent.

In [15]:
program = build_system(interactive=True).compile()
waiting = program.run(CASE, timeout_s=40)
assert waiting.status == "paused"
print("Question:", waiting.interrupts)
restored = toolkit.RunResult(**waiting.to_dict())
finished = program.resume(restored, True)
assert finished.ok and finished.data["refund"] == 120
print("Output:", finished.data)
print(finished.lineage(name="R-17").render())
print("For another agent:", finished.lineage().to_prompt_context())

Question: ({'id': '82fe5a3c5adf8d2cc063bddb1dc47a6e', 'value': {'question': 'Approve partial refund?', 'refund': 120, 'order_id': 'R-17'}},)


Output: {'order_id': 'R-17', 'refund': 120, 'currency': 'MXN', 'missing_units': 1, 'reason': 'Difference between packing slip and received item list', 'sources': ['packing-slip', 'customer-report'], 'previous_episodes': 0, 'status': 'approved'}
R-17: OK
{"order_id": "R-17", "refund": 120, "currency": "MXN", "missing_units": 1, "reason": "Difference between packing slip and received item list", "sources": ["packing-slip", "customer-report"], "previous_episodes": 0, "status": "approved"}
- refunds: completed [run-efbe41f3d22f412380543b4dcefb6ba3]
- normalize: completed [run-b61cb8bc2fa64e0e9d96cdb232c0ed73]
- normalize: completed [tool-22d67f05235941118dd5c75d2656e554]
- classify: classify -> {"order_id": "R-17", "amount": 240, "total_units": 2, "missing_units": 1, "reason": "Difference between packing slip and received item list", "sources": ["packing-slip", "customer-report"] [run-ca401f8a71ad4646bcab13bd10787f47]
- eval: classify -> {"order_id": "R-17", "amount": 240, "total_units": 2

## model_artifact: identity and provenance

Declare the weights that would be served. This does not download, train, load or validate them. main is a mutable revision; pin revision and hashes before real validation.

In [16]:
artifact = toolkit.model_artifact(
    "Qwen/Qwen2.5-0.5B-Instruct",
    revision="main",
    metadata={"purpose": "offline serving declaration; not validated weights"},
)
print(artifact.inspect())

{'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'base_model': None, 'adapter_path': None, 'tokenizer': None, 'revision': 'main', 'quantization': None, 'metadata': {'purpose': 'offline serving declaration; not validated weights'}}


## model_server: a serving plan without startup

Construct configuration, command and endpoint. Offline execution checks the plan and key redaction without start or health calls. Separate tests simulate start/health/stop without processes or network. Real startup requires vLLM, weights and a compatible GPU environment and remains unverified.

In [17]:
server = toolkit.model_server(
    artifact,
    profile="custom",
    served_model_name="quote-model",
    gpu_memory_utilization=0.5,
    max_model_len=2048,
    max_num_seqs=1,
    enable_auto_tool_choice=False,
    tool_call_parser=None,
    api_key="example-local-token",
)
connection = toolkit.runtime(
    provider="vllm-runtime",
    model=server.endpoint.model,
    endpoint=server.endpoint.url,
    api_key="example-local-token",
    max_tokens=128,
)
assert server.endpoint.owned is False
assert "example-local-token" not in str(server.inspect())
print("Redacted command:", server.command)
print("Planned endpoint:", server.endpoint)
print("Planned connection, without inference:", connection.inspect())

Redacted command: ('vllm', 'serve', 'Qwen/Qwen2.5-0.5B-Instruct', '--host', '127.0.0.1', '--port', '8000', '--served-model-name', 'quote-model', '--gpu-memory-utilization', '0.5', '--max-model-len', '2048', '--max-num-seqs', '1', '--generation-config', 'vllm', '--api-key', '[REDACTED]')
Planned endpoint: _Endpoint(url='http://127.0.0.1:8000/v1', model='quote-model', backend='vllm', api_key_configured=True, pid=None, owned=False)
Planned connection, without inference: {'provider': 'vllm-runtime', 'model': 'quote-model', 'region': None, 'endpoint': 'http://127.0.0.1:8000/v1', 'api_key_configured': True, 'session_configured': False, 'authentication_mode': 'provider-configuration', 'timeout_s': 60.0, 'max_retries': 0, 'max_tool_calls': 5, 'max_turns': 6, 'max_tokens': 128, 'backoff_s': 0.0, 'max_concurrency': 1}


### Prepared GPU environment only; not executed here

```python
server.start()
try:
    assert server.health().status == "healthy"
    # connection already targets the declared endpoint and model.
    # Inference requires explicit campaign authorization and budget.
finally:
    server.stop()
```

This review is offline and does not publish an artifact. Historical OpenAI evidence is archived as dev1. Coverage terminology distinguishes factory invocation, a simulated lifecycle and real infrastructure availability.